In [36]:
# new dataset with updated features 

# find the data file 
data_path = dir.parent / 'Data' / 'Dominant_feat.csv'

# load data in object 
df2 = pd.read_csv(data_path)

# view result 
df2.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,Embarked_644,Embarked_C,Embarked_Q,Sex_female,Cabin_encoded,interaction4,interaction5,interaction6,interaction7,Ratio1,Ratio4,Ratio5
0,0,3,-0.581659,1,0,-0.879741,False,False,False,False,0.303217,0.511709,-2.639222,-0.000000,-0.879741,0.661171,-0.000000,-1.136699
1,1,1,0.649327,1,0,1.361220,False,True,False,True,0.382889,0.883877,1.361220,1.361220,0.000000,0.477018,0.734635,0.000000
2,1,3,-0.273913,0,0,-0.798540,False,False,False,True,0.298863,0.218730,-2.395620,-0.798540,-0.000000,0.343017,-1.252285,-0.000000
3,1,1,0.418517,1,0,1.062038,False,False,False,True,0.000000,0.444481,1.062038,1.062038,0.000000,0.394070,0.941586,0.000000
4,0,3,0.418517,0,0,-0.784179,False,False,False,False,0.303215,-0.328192,-2.352538,-0.000000,-0.784179,-0.533701,-0.000000,-1.275219


In [37]:
# Training Phase 
# import neccessary library from sklearn
from sklearn.model_selection import train_test_split

# inputs/features (rop target feature to avoid data leakage)
X = df2.drop(columns=['Survived'])

# Target variable 
Y = df2['Survived']

# 70/30 split 
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size= 0.3, shuffle= True, random_state= 42)

#Reset indexs 
X_train = X_train.reset_index(drop = True)
Y_train = Y_train.reset_index(drop = True)

# save training dataset 
X_train.to_csv('X_train.csv')
Y_train.to_csv('Y_train.csv')


In [38]:
# Modeling Phase (Classification)
# import sklearn libraries for downstream modeling workflow 
from sklearn.metrics import confusion_matrix, classification_report # numeric performance indicators

## K-NearestNeighbors Algorithm

Metrics: 
- Recall: 73% of the positive cases we're identified and caught by the model, reducing false negatives
- Precision: 79% of positive cases we're accurately predicted by the model, reducing false posiitives 
- F1 Score: The harmonic mean of the two metric are 81% signaling a usefulness in the model predictions for real world implementation

Key Insights: Model peforms 5% better when it focuses on capturing the best recall score! 


In [39]:
# Baseline Model (KNN) uses distances to classify which class a data point belongs in 

from sklearn.neighbors import KNeighborsClassifier 

# instantiation process
KNN = KNeighborsClassifier(n_neighbors=5)

# fit the data on the model 
KNN.fit(X_train, Y_train)

# predictions
KNN_pred = KNN.predict(X_test)

# performance results 
print(f'Classification Report: {classification_report(Y_test, KNN_pred)}')
print(f'Confusion Matric: {confusion_matrix(Y_test, KNN_pred)}')

Classification Report:               precision    recall  f1-score   support

           0       0.80      0.88      0.84       157
           1       0.80      0.68      0.74       111

    accuracy                           0.80       268
   macro avg       0.80      0.78      0.79       268
weighted avg       0.80      0.80      0.80       268

Confusion Matric: [[138  19]
 [ 35  76]]


In [40]:
# Find the best optimal settings for K-Nearest Neighbors 
# import gridsearchcv to locate the best paramaters for the model
from sklearn.model_selection import GridSearchCV

params = {'n_neighbors': [1, 5, 10, 15, 20], 
          'weights': ['uniform', 'distance'], 
          'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'], 
          'leaf_size': [10, 20, 30, 40], }
# instantiation process
KNN_best = GridSearchCV(estimator = KNN, param_grid= params, cv = 5, verbose = 5, scoring = 'recall')

# fit data on the model 
KNN_best.fit(X_train, Y_train)

# predictions 
KNN_pred = KNN_best.predict(X_test)

# performance results
print(f'Classification Report:{classification_report(Y_test, KNN_pred)}')
print(f'Best Settings: {(KNN_best.best_estimator_)}')

Fitting 5 folds for each of 160 candidates, totalling 800 fits
[CV 1/5] END algorithm=auto, leaf_size=10, n_neighbors=1, weights=uniform;, score=0.723 total time=   0.0s
[CV 2/5] END algorithm=auto, leaf_size=10, n_neighbors=1, weights=uniform;, score=0.717 total time=   0.0s
[CV 3/5] END algorithm=auto, leaf_size=10, n_neighbors=1, weights=uniform;, score=0.478 total time=   0.0s
[CV 4/5] END algorithm=auto, leaf_size=10, n_neighbors=1, weights=uniform;, score=0.696 total time=   0.0s
[CV 5/5] END algorithm=auto, leaf_size=10, n_neighbors=1, weights=uniform;, score=0.696 total time=   0.0s
[CV 1/5] END algorithm=auto, leaf_size=10, n_neighbors=1, weights=distance;, score=0.723 total time=   0.0s
[CV 2/5] END algorithm=auto, leaf_size=10, n_neighbors=1, weights=distance;, score=0.717 total time=   0.0s
[CV 3/5] END algorithm=auto, leaf_size=10, n_neighbors=1, weights=distance;, score=0.478 total time=   0.0s
[CV 4/5] END algorithm=auto, leaf_size=10, n_neighbors=1, weights=distance;, s

## logistic Regression
Performance Metrics: 
- Recall: 73% of positve cases were caught by the model, signaling a reduction in false negatives 
- Precision: 84% of positive cases were accurately predicted, reducing the amount of false positives
- Fl Score: The harmonic mean was calcuated ay 78%, signaling a level of usefulness for the model


In [41]:
# Let's use a simpler model 

from sklearn.linear_model import LogisticRegression

# begin instantiantion process
lr = LogisticRegression(max_iter= 100)

# fit data on model
lr.fit(X_train, Y_train)

# predictions 
lr_pred = lr.predict(X_test)

# performance metrrics
print(f'Classification Report {classification_report(Y_test, lr_pred)}')
print(f'Confusion matric {confusion_matrix(Y_test, lr_pred)}')


Classification Report               precision    recall  f1-score   support

           0       0.83      0.90      0.86       157
           1       0.84      0.73      0.78       111

    accuracy                           0.83       268
   macro avg       0.83      0.82      0.82       268
weighted avg       0.83      0.83      0.83       268

Confusion matric [[142  15]
 [ 30  81]]


In [42]:
# Let's find the best hyperparameter settings for the model 

# parameters
params = {'max_iter': [100, 500, 1000, 2000]}

# instantiantion process 
lr_best = GridSearchCV(estimator= lr, param_grid= params, cv = 5, verbose= 5, n_jobs= -1)

# fit data on model
lr_best.fit(X_train, Y_train)

# predictions
lr_pred = lr_best.predict(X_test)

# performance metrics 
print(f'Classification Report C{classification_report(Y_test, lr_pred)}')
print(f'Confusion Matrix {confusion_matrix(Y_test, lr_pred)}')
print(f'Best settings {lr_best.best_estimator_}')

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Classification Report C              precision    recall  f1-score   support

           0       0.83      0.90      0.86       157
           1       0.84      0.73      0.78       111

    accuracy                           0.83       268
   macro avg       0.83      0.82      0.82       268
weighted avg       0.83      0.83      0.83       268

Confusion Matrix [[142  15]
 [ 30  81]]
Best settings LogisticRegression()


## Support Vector Machines

Peformance Metrics:
- Recall: 70% of postives cases were caught by the model, reducing some level false negatives. 
- precision: 80% of positive cases were accurately predicted by the model, reduing false positves.
- The harmonic mean (f1) outputted a 75%, suggesting the model has a moderate use case in predicting  

Key Insights: 
- In the process of utilizing GridSeacrchCv, the model had trouble converging. High numerical SVM parameters work extremely hard to achieve 100% accuracy on the training data. Thus, giving the model a hard time to come up with a prediction within a certiain timeframe even with the implementation of n_jobs (a parameters that tells the model to use all cpu's inside of a computer). The optimal solution in this case was to envoke HavlingGridSearchCV (a derivitive of GridSearchCV with the ability to use a subset of data points, leading to faster predictions and best setting outputs)

In [43]:
from sklearn.svm import SVC # support vector model 
# instantiation process
svm = SVC()

# fit the data on the model
svm.fit(X_train, Y_train)

# predictios
Y_predictions = svm.predict(X_test)

# predictions results
Y_predictions

# Evaluate model performance results 
print(f'Classification Report {classification_report(Y_test, Y_predictions)}')
print(f'Confusion Matrix {confusion_matrix(Y_test, Y_predictions)}')

Classification Report               precision    recall  f1-score   support

           0       0.76      0.92      0.83       157
           1       0.83      0.59      0.69       111

    accuracy                           0.78       268
   macro avg       0.80      0.75      0.76       268
weighted avg       0.79      0.78      0.77       268

Confusion Matrix [[144  13]
 [ 46  65]]


In [44]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV 

# parameters
params = {'gamma': [0.1, 1, 5, 10],
          "C": [1, 10, 100], 
          'kernel': ['rbf', 'linear']}

# Grid Search settings 
svm_best = HalvingGridSearchCV(estimator = svm, param_grid = params, cv =5, factor = 2, n_jobs = -1, verbose = 5)

# fit data onto model
svm_best.fit(X_train, Y_train)

# predictions
svm_pred = svm_best.predict(X_test)

# performance result 
print(f'Classification Report: {classification_report(Y_test, svm_pred)}')
print(f'confusion Matrix: {confusion_matrix(Y_test, svm_pred)}')
print(f'Best Settings: {svm_best.best_estimator_}')

n_iterations: 5
n_required_iterations: 5
n_possible_iterations: 5
min_resources_: 38
max_resources_: 623
aggressive_elimination: False
factor: 2
----------
iter: 0
n_candidates: 24
n_resources: 38
Fitting 5 folds for each of 24 candidates, totalling 120 fits


C:\Users\dmiracju\AppData\Roaming\Python\Python313\site-packages\numpy\ma\core.py:2896: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,
C:\Users\dmiracju\AppData\Roaming\Python\Python313\site-packages\numpy\ma\core.py:2896: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


----------
iter: 1
n_candidates: 12
n_resources: 76
Fitting 5 folds for each of 12 candidates, totalling 60 fits
----------
iter: 2
n_candidates: 6
n_resources: 152
Fitting 5 folds for each of 6 candidates, totalling 30 fits
----------
iter: 3
n_candidates: 3
n_resources: 304
Fitting 5 folds for each of 3 candidates, totalling 15 fits


C:\Users\dmiracju\AppData\Roaming\Python\Python313\site-packages\numpy\ma\core.py:2896: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


----------
iter: 4
n_candidates: 2
n_resources: 608
Fitting 5 folds for each of 2 candidates, totalling 10 fits


C:\Users\dmiracju\AppData\Roaming\Python\Python313\site-packages\numpy\ma\core.py:2896: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,


Classification Report:               precision    recall  f1-score   support

           0       0.80      0.86      0.83       157
           1       0.78      0.70      0.74       111

    accuracy                           0.79       268
   macro avg       0.79      0.78      0.79       268
weighted avg       0.79      0.79      0.79       268

confusion Matrix: [[135  22]
 [ 33  78]]
Best Settings: SVC(C=100, gamma=0.1, kernel='linear')


## Random Forest Classification
- Performance Metrics: 
- Recall: 67% of positive cases were caught and predicted by the model, which entails the model is weak/useless in predicting 
- Precision 83% of positive cases were accurately predicted by the model, reducing the amouf of false positives. 
- F1 Score: The harmonic mean calculated between the recall and precision adjusted for 73%, signaling a slight usefulness of the model.  

In [45]:
# Random Forest Classification 
from sklearn.ensemble import RandomForestClassifier 

# instantiation process
rf = RandomForestClassifier(n_estimators= 100)

# fit data onto model 
rf.fit(X_train, Y_train)

# predictions
rf_pred = rf.predict(X_test)

# peformance evaluation
print(f'Classification Repor: {classification_report(Y_test, rf_pred)}')
print(f'Confusion Matrix: {confusion_matrix(Y_test, rf_pred)}')

Classification Repor:               precision    recall  f1-score   support

           0       0.80      0.87      0.83       157
           1       0.79      0.68      0.73       111

    accuracy                           0.79       268
   macro avg       0.79      0.78      0.78       268
weighted avg       0.79      0.79      0.79       268

Confusion Matrix: [[137  20]
 [ 35  76]]


In [46]:
# Find best parameters for the model 
# params 
params = {'n_estimators': [100, 200, 300, 400, 500,], 
          'max_depth': [None], 
          'min_samples_split': [2, 5, 10, 0.05],
          'min_samples_leaf': [1, 2, 4, 0.5],
          'max_features': ['sqrt', 'log2'],
          'random_state': [42], 
          'n_jobs': [-1]}

# instantation process 
rf_best = GridSearchCV(estimator= rf, param_grid= params, cv = 5, verbose = 5, n_jobs = -1, error_score= 'raise')
# fit data onto model 
rf_best.fit(X_train, Y_train)

# predicitions
rf_pred = rf_best.predict(X_test)

# performance summary 
print(f'Classification Report: {classification_report(Y_test, rf_pred)}')
print(f'Confusion Matrix: {confusion_matrix(Y_test, rf_pred)}')
print(f'Best Settings: {rf_best.best_estimator_}')

Fitting 5 folds for each of 160 candidates, totalling 800 fits
Classification Report:               precision    recall  f1-score   support

           0       0.80      0.89      0.84       157
           1       0.81      0.68      0.74       111

    accuracy                           0.80       268
   macro avg       0.80      0.79      0.79       268
weighted avg       0.80      0.80      0.80       268

Confusion Matrix: [[139  18]
 [ 35  76]]
Best Settings: RandomForestClassifier(min_samples_split=10, n_estimators=300, n_jobs=-1,
                       random_state=42)


# XGboost Algorithm 
Performance Metrics: 
- Recall: 70% of cases were caught by the model, reducing false negative in process.
- Precision: 80% of cases were accurately predicted by the model, which led to the reduction in false positives. 
- F1: The harmonic mean of recall and precision adjusted for 75% signaling a moderate usefullness in real world application

In [47]:
# Let's use a boosted algorithm to capture omore accuacry and performance 

# import package 
import xgboost as xgb 

# intialize xgboostclassifier
xgb = xgb.XGBClassifier(n_estimators = 100)
# fit data on boosted algo
xgb.fit(X_train, Y_train)

# predictions
xgb_pred = xgb.predict(X_test)

# performance metrics 
print(f'Classification Report {classification_report(Y_test, xgb_pred)}')
print(f'Confusin Matrix {confusion_matrix(Y_test, xgb_pred)}')



Classification Report               precision    recall  f1-score   support

           0       0.81      0.88      0.84       157
           1       0.80      0.70      0.75       111

    accuracy                           0.81       268
   macro avg       0.81      0.79      0.80       268
weighted avg       0.81      0.81      0.80       268

Confusin Matrix [[138  19]
 [ 33  78]]


In [48]:
# Let's find the best settings for the boosted algorithm

# parameters 
params = {'n_estimators': [100, 200, 500, 1000], 
          'learning_rate': [0.1, 0.2, 5, 10], 
          'max_depth': [5, 10, 20, 30, 40]}

# intitalize gridsearch 
xgb_best = GridSearchCV(estimator= xgb, param_grid= params, cv = 5, verbose = 5, n_jobs =-1) 

# fit data on model
xgb_best.fit(X_train, Y_train)

# predcitions
xgb_pred = xgb.predict(X_test)

# evaluation metrics 
print(f'Classification Report {classification_report(Y_test, xgb_pred)}')
print(f'Confusion Matirx {confusion_matrix(Y_test, xgb_pred)}')
print(f'Best settings {xgb_best.best_estimator_}')


Fitting 5 folds for each of 80 candidates, totalling 400 fits
Classification Report               precision    recall  f1-score   support

           0       0.81      0.88      0.84       157
           1       0.80      0.70      0.75       111

    accuracy                           0.81       268
   macro avg       0.81      0.79      0.80       268
weighted avg       0.81      0.81      0.80       268

Confusion Matirx [[138  19]
 [ 33  78]]
Best settings XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=N